# Capstone — Build, Train & Report a Deep Learning Model

This is **your** project. Across the course you built — by hand, on toy numbers in
the sandbox — tensors, the training loop, CNNs, RNNs/LSTMs, regularization, and the
habit of reading loss curves. Now you put it all together on a **real dataset you
choose**, and you are judged on a **held-out test set** you only touch once.

Unlike the weekly labs, this notebook is **mostly scaffold**. It gives you the
boring-but-essential spine — `device`/seed setup, `train()` / `evaluate()` /
`plot_curves()`, and a one-shot test-evaluation cell — so you can spend your effort
on the part that matters: the **model** and the **experiments**. Everything marked
**🔧 TODO** is yours to fill in. The notebook will not learn anything until you do.

> **Runtime → Change runtime type → T4 GPU** before you start. Then run the setup
> cell and confirm it prints `cuda`.

---

## Choose your track

Pick **one** track below and uncomment its data-loading stub in Beat 3. All three
use a *real* public dataset and target a model family you already met in the course.
You are free to invent your own task instead (any real dataset + a model you train)
— these three are just well-scoped starting points.

| Track | Dataset (real, auto-downloaded) | Model you build | Beat this baseline |
|---|---|---|---|
| **A · Vision** | **CIFAR-10** — 60k 32×32 colour photos, 10 classes | a **CNN** (Conv → ReLU → Pool …) | majority class = **10%**; aim **> 65%** test accuracy |
| **B · Text** | **Names-by-language** — ~20k surnames, 18 languages (PyTorch's char-RNN dataset) | a char-level **LSTM** classifier | majority class (English) ≈ **27%**; aim **> 70%** |
| **C · Tabular** | **Penguins** — 344 rows, 7 features, 3 species (seaborn) | an **MLP** | majority class ≈ **44%**; aim **> 95%** |

Each track is sized to train in **under ~5 minutes on a T4**. The spine below is
**track-agnostic**: write your dataset loader and model so that `train_loader`,
`val_loader`, `test_loader`, and `model` exist, and the generic `train()` /
`evaluate()` helpers just work.

> **The one rule that defines this capstone:** *tune on validation, report on test.*
> Choose your model, learning rate, epochs, and regularization using the **validation**
> set. Touch the **test** set exactly **once**, at the very end, for the number you
> report. Peeking at test while tuning turns it into a second validation set and
> quietly inflates your result — the rubric checks for this.

## 1 · Setup, GPU check & the spine

Run this once. It imports everything, fixes the seed so your runs are reproducible
(the rubric expects a fixed seed), picks the GPU, and defines the **spine** you'll
reuse no matter which track you chose:

- `accuracy(logits, y)` — exact-match accuracy for a batch of logits.
- `train(...)` — the generic five-step training loop with a validation pass each
  epoch; returns the per-epoch `history`.
- `evaluate(...)` — a no-grad forward pass returning `(loss, accuracy)`.
- `plot_curves(history)` — train-vs-val loss and accuracy, the diagnostic you learned
  to read in the labs.

These are deliberately model-agnostic: they assume a classifier whose `model(x)`
returns class **logits** and whose loader yields `(x, y)` batches. That covers all
three tracks. (Doing a regression or a language model instead? Swap
`CrossEntropyLoss`/accuracy for MSE/perplexity — the loop shape is identical.)

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

# --- reproducibility: fix the seed so your reported numbers reproduce -------
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- device: train on the GPU if Colab gave you one -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")


@torch.no_grad()
def accuracy(logits, y):
    """Fraction correct for a batch of class logits vs integer labels."""
    return (logits.argmax(dim=-1) == y).float().mean().item()


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    """One no-grad pass over `loader`; returns (mean loss, accuracy)."""
    model.eval()                                  # dropout off, BN uses running stats
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)         # same device as the model
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(dim=-1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


def train(model, train_loader, val_loader, *, epochs=10, lr=1e-3,
          weight_decay=0.0, loss_fn=None, max_grad_norm=None):
    """The five-step loop (forward, loss, zero_grad, backward, step) with a
    validation pass each epoch. Returns a per-epoch history dict."""
    model = model.to(device)                      # move the model once, before the loop
    loss_fn = loss_fn or nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:                 # the five steps, per batch
            x, y = x.to(device), y.to(device)     # data on the SAME device as the model
            logits = model(x)                     # 1. forward
            loss = loss_fn(logits, y)             # 2. loss
            optimizer.zero_grad()                 # 3. clear last batch's grads
            loss.backward()                       # 4. backprop
            if max_grad_norm is not None:         # clip for RNNs/LSTMs (Track B)
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()                      # 5. update the weights

        tr_loss, tr_acc = evaluate(model, train_loader, loss_fn)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)
        print(f"epoch {epoch + 1:2d}/{epochs}  "
              f"train_loss={tr_loss:.3f} acc={tr_acc:.2%}  |  "
              f"val_loss={va_loss:.3f} acc={va_acc:.2%}")
    return history


def plot_curves(history):
    """Train vs. validation loss and accuracy — read the gap (labs L5/L11)."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(epochs, history["train_loss"], "-o", label="train")
    ax1.plot(epochs, history["val_loss"], "-o", label="val")
    ax1.set(xlabel="epoch", ylabel="loss", title="Loss")
    ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(epochs, history["train_acc"], "-o", label="train")
    ax2.plot(epochs, history["val_acc"], "-o", label="val")
    ax2.set(xlabel="epoch", ylabel="accuracy", title="Accuracy")
    ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


print("spine ready: accuracy(), evaluate(), train(), plot_curves()")

## 2 · 🔧 Load your real dataset (with a train / val / **test** split)

**Uncomment exactly one block below** (your chosen track). Each one downloads a real
dataset over the internet (Colab has it), builds a model-agnostic train/val/test
split, and exposes the **same four names** the spine expects:

```
train_loader   val_loader   test_loader   classes
```

plus whatever your model needs (`in_features`, `vocab_size`, `n_classes`, …). Keeping
the interface identical across tracks is what lets the generic `train()` / `evaluate()`
helpers work unchanged.

> **🔧 Your turn — the split is the load-bearing part.** Carve out a **test** set and
> *do not look at it* until the final cell. Tune everything on **val**. Each stub
> below does a clean 3-way split with the fixed `SEED`; read it, and if you bring your
> own dataset, replicate that discipline (no overlap, no peeking).

In [ ]:
# ============================================================================
# TRACK A · VISION — CIFAR-10 (real 32x32 colour photos, 10 classes)
# Build a CNN. This block is ACTIVE so the notebook runs end-to-end out of the
# box; comment it out if you pick Track B or C.
# ============================================================================
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

# What's real/messy: natural photos with backgrounds, lighting, and pose
# variation. Some classes look alike (cat vs dog, automobile vs truck) -- your
# confusion matrix later will show exactly which.
norm = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
train_tf = transforms.Compose([                 # augment ONLY the training set
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(), norm,
])
eval_tf = transforms.Compose([transforms.ToTensor(), norm])   # never augment val/test

# CIFAR's official split is 50k train + 10k test. We hold the 10k test out
# untouched, and carve a validation set off the training data.
full_train = datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
test_set = datasets.CIFAR10("./data", train=False, download=True, transform=eval_tf)

# Subset to train fast on a T4 (~3-4 min). Use all 50k for a stronger result.
SUBSET = 10000
g = torch.Generator().manual_seed(SEED)
idx = torch.randperm(len(full_train), generator=g)[:SUBSET].tolist()
n_val = SUBSET // 5
train_idx, val_idx = idx[n_val:], idx[:n_val]
train_set = Subset(full_train, train_idx)
# val must NOT be augmented -> point it at a non-augmented copy of the data
val_base = datasets.CIFAR10("./data", train=True, download=True, transform=eval_tf)
val_set = Subset(val_base, val_idx)

batch_size = 128
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)
classes = full_train.classes                    # ['airplane', 'automobile', ...]
print(f"train={len(train_set)} val={len(val_set)} test={len(test_set)} "
      f"classes={len(classes)}")

In [ ]:
# ============================================================================
# TRACK B · TEXT — Names-by-language (real surnames, 18 languages)
# Build a char-level LSTM classifier. UNCOMMENT this whole block to use it,
# and comment out Track A above.
# ============================================================================
# import os, glob, unicodedata, string, zipfile, urllib.request
# from torch.utils.data import DataLoader, TensorDataset, random_split
# from torch.nn.utils.rnn import pad_sequence
#
# # Real/messy: PyTorch's char-RNN dataset -- one .txt per language, one surname
# # per line, accented characters, and very imbalanced classes (English/Russian
# # dominate). Stable URL, ~2.8 MB.
# URL = "https://download.pytorch.org/tutorial/data.zip"
# if not os.path.exists("data/names"):
#     urllib.request.urlretrieve(URL, "data.zip")
#     with zipfile.ZipFile("data.zip") as z:
#         z.extractall(".")
#
# all_letters = string.ascii_letters + " .,;'"
# stoi = {c: i + 1 for i, c in enumerate(all_letters)}   # 0 reserved for <pad>
# vocab_size = len(stoi) + 1
#
# def to_ascii(s):                                # strip accents -> ascii
#     return "".join(c for c in unicodedata.normalize("NFD", s)
#                     if unicodedata.category(c) != "Mn" and c in all_letters)
#
# names, labels, classes = [], [], []
# for path in sorted(glob.glob("data/names/*.txt")):
#     lang = os.path.splitext(os.path.basename(path))[0]
#     classes.append(lang)
#     with open(path, encoding="utf-8") as f:
#         for line in f:
#             name = to_ascii(line.strip())
#             if name:
#                 names.append(torch.tensor([stoi[c] for c in name]))
#                 labels.append(len(classes) - 1)
# n_classes = len(classes)
#
# X = pad_sequence(names, batch_first=True, padding_value=0)   # (N, max_len)
# y = torch.tensor(labels)
# full = TensorDataset(X, y)
# n_test = len(full) // 5
# n_val = len(full) // 5
# n_train = len(full) - n_val - n_test
# g = torch.Generator().manual_seed(SEED)
# train_set, val_set, test_set = random_split(full, [n_train, n_val, n_test], generator=g)
#
# batch_size = 64
# train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
# val_loader = DataLoader(val_set, batch_size=batch_size)
# test_loader = DataLoader(test_set, batch_size=batch_size)
# print(f"train={n_train} val={n_val} test={n_test} "
#       f"classes={n_classes} vocab={vocab_size}")

In [ ]:
# ============================================================================
# TRACK C · TABULAR — Penguins (real measurements, 3 species)
# Build an MLP. UNCOMMENT this whole block to use it, and comment out Track A.
# ============================================================================
# import pandas as pd
# import seaborn as sns
# from sklearn.preprocessing import StandardScaler
# from torch.utils.data import DataLoader, TensorDataset, random_split
#
# # Real/messy: a real ecology dataset with MISSING values (rows with NaN) and
# # categorical columns (island, sex) you must encode. 344 rows, 3 species.
# df = sns.load_dataset("penguins").dropna().reset_index(drop=True)
# target = "species"
# classes = sorted(df[target].unique())          # ['Adelie', 'Chinstrap', 'Gentoo']
# y = torch.tensor([classes.index(s) for s in df[target]])
#
# num_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
# cat_cols = ["island", "sex"]
# X_df = pd.concat([df[num_cols], pd.get_dummies(df[cat_cols], dtype=float)], axis=1)
# in_features = X_df.shape[1]
# n_classes = len(classes)
#
# full = TensorDataset(torch.tensor(X_df.values, dtype=torch.float32), y)
# n_test = len(full) // 5
# n_val = len(full) // 5
# n_train = len(full) - n_val - n_test
# g = torch.Generator().manual_seed(SEED)
# train_set, val_set, test_set = random_split(full, [n_train, n_val, n_test], generator=g)
#
# # Standardize numeric features: FIT ON TRAIN ONLY, reuse on val/test (no leakage).
# train_X = full.tensors[0][train_set.indices]
# scaler = StandardScaler().fit(train_X[:, :len(num_cols)].numpy())
# def scale(ds):
#     xb = full.tensors[0][ds.indices].clone()
#     xb[:, :len(num_cols)] = torch.tensor(
#         scaler.transform(xb[:, :len(num_cols)].numpy()), dtype=torch.float32)
#     return TensorDataset(xb, full.tensors[1][ds.indices])
# train_set, val_set, test_set = scale(train_set), scale(val_set), scale(test_set)
#
# batch_size = 16
# train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
# val_loader = DataLoader(val_set, batch_size=batch_size)
# test_loader = DataLoader(test_set, batch_size=batch_size)
# print(f"train={n_train} val={n_val} test={n_test} "
#       f"classes={n_classes} features={in_features}")

## 3 · Look at the data

Seeing the data *is* the real-dataset experience — it's where you notice the class
imbalance, the messy rows, the look-alike classes. The cell below shows a class
histogram + a sample for **Track A**. If you switched tracks, adapt it: show a few
raw surnames per language (B) or `df.head()` and `df.describe()` (C).

In [ ]:
# EDA for the ACTIVE track (A · CIFAR-10). Adapt for B (print sample names per
# class) or C (df.head(), df["species"].value_counts(), a pairplot).
import collections

# 1. Class histogram of the training split -- is it balanced?
counts = collections.Counter(int(train_set[i][1]) for i in range(len(train_set)))
plt.figure(figsize=(9, 3))
plt.bar([classes[k] for k in sorted(counts)], [counts[k] for k in sorted(counts)])
plt.xticks(rotation=45, ha="right"); plt.ylabel("count")
plt.title("Training-set class balance"); plt.tight_layout(); plt.show()
print("majority-class baseline (train):",
      f"{max(counts.values()) / sum(counts.values()):.1%}")

# 2. A few sample images with their labels.
inv = transforms.Normalize((-0.4914 / 0.2470, -0.4822 / 0.2435, -0.4465 / 0.2616),
                            (1 / 0.2470, 1 / 0.2435, 1 / 0.2616))
fig, axes = plt.subplots(1, 6, figsize=(11, 2.2))
for ax, i in zip(axes, range(6)):
    img, label = train_set[i]
    ax.imshow(inv(img).permute(1, 2, 0).clamp(0, 1).numpy())
    ax.set_title(classes[label], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 4 · 🔧 Build your model

This mirrors what you built from scratch in the labs — **read it, and try writing it
yourself before running.** The scaffold gives a working starter for each track so the
notebook trains immediately; the rubric expects *you* to design and justify the
architecture, so treat the starter as a baseline to **improve on**, not a final answer.

> **🔧 Your turn — define `model`.** Uncomment the starter for your track (the **CNN**
> for A is active by default). Then make it yours: depth, width, dropout, an extra conv
> block, a second LSTM layer — whatever you'll defend in your report. `model(x)` must
> return class **logits**; the spine does the rest.

In [ ]:
# ---- TRACK A · CNN (active) -- Conv -> ReLU -> Pool, twice, then a linear head.
# Mirrors the L6 recipe; CIFAR is 3-channel 32x32 (-> 16x16 -> 8x8).
class CNN(nn.Module):
    def __init__(self, n_classes=10, p_drop=0.25):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                  # 32x32 -> 16x16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                  # 16x16 -> 8x8
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
            nn.Dropout(p_drop),                               # regularization (L5)
            nn.Linear(128, n_classes),
        )

    def forward(self, x):                  # x: (batch, 3, 32, 32)
        return self.head(self.features(x))

model = CNN(n_classes=len(classes))

# ---- TRACK B · LSTM classifier -- embedding -> LSTM -> head off the last state.
# class NameLSTM(nn.Module):
#     def __init__(self, vocab_size, n_classes, emb=32, hidden=64):
#         super().__init__()
#         self.embed = nn.Embedding(vocab_size, emb, padding_idx=0)  # 0 = <pad>
#         self.lstm = nn.LSTM(emb, hidden, batch_first=True)
#         self.head = nn.Linear(hidden, n_classes)
#     def forward(self, x):                  # x: (batch, time) of char ids
#         h, (h_n, c_n) = self.lstm(self.embed(x))
#         return self.head(h_n[-1])          # last hidden state -> class logits
# model = NameLSTM(vocab_size, n_classes)    # train with max_grad_norm=1.0

# ---- TRACK C · MLP -- a couple of hidden layers with dropout.
# class MLP(nn.Module):
#     def __init__(self, in_features, n_classes, hidden=32, p_drop=0.2):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(in_features, hidden), nn.ReLU(), nn.Dropout(p_drop),
#             nn.Linear(hidden, hidden), nn.ReLU(),
#             nn.Linear(hidden, n_classes),
#         )
#     def forward(self, x):
#         return self.net(x)
# model = MLP(in_features, n_classes)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"trainable parameters: {n_params:,}")

## 5 · 🔧 Train

The five-step loop lives in `train()` (Beat 1) — you just call it. Keep the first run
**fast** (few epochs / the subset) so you get a signal in minutes, then scale up once
the wiring works.

> **🔧 Your turn — run it, then tune to beat the baseline.** Call `train(...)`, read
> the printed `val_acc`, and iterate: learning rate, epochs, `weight_decay`, dropout,
> model size, augmentation. **For Track B (LSTM)** pass `max_grad_norm=1.0` to clip
> exploding gradients. Tune on **val** only — the test set stays sealed until Beat 7.

> **Cheapest sanity check first:** if a fresh model won't drive the loss down at all,
> *overfit one tiny batch* (high lr, many steps, no dropout). If it can't memorise a
> handful of examples, the bug is in the wiring (a wrong shape, a frozen layer), not
> the hyperparameters — fix that before a long run.

In [ ]:
# 🔧 Your main run. Tune these on VALIDATION; do not touch test yet.
history = train(
    model,
    train_loader,
    val_loader,
    epochs=8,            # 🔧 enough to see val accuracy plateau
    lr=1e-3,             # 🔧 if loss diverges/NaN, drop 10x; if it crawls, raise it
    weight_decay=1e-4,   # 🔧 L2 regularization (Track B: also max_grad_norm=1.0)
)

### 🔧 (Optional) Diagnose & fix a broken run

This cell is **isolated** — it trains its own throwaway model, so it never touches your
real `model` or `history` above. It has a **planted bug**: the classic training-loop
mistake from the labs. Run it, watch the loss *refuse to fall*, then **find and fix the
one missing line** and watch it learn.

> **🔧 Your turn:** what one step of the five-step heartbeat is missing? Why does
> leaving it out make the gradient grow batch-over-batch and the loss lurch? Fix it,
> re-run, and confirm the loss now drops.

In [ ]:
# 🔧 BROKEN ON PURPOSE -- find the missing line. (Isolated: uses its own model.)
import copy


def overfit_one_batch_demo():
    torch.manual_seed(SEED)
    x, y = next(iter(train_loader))
    x, y = x.to(device), y.to(device)
    probe = copy.deepcopy(model)                    # a fresh, independent copy
    for m in probe.modules():                       # re-init so it starts untrained
        if hasattr(m, "reset_parameters"):
            m.reset_parameters()
    probe = probe.to(device)
    opt = torch.optim.SGD(probe.parameters(), lr=0.1)   # plain SGD, high lr (overfit 1 batch)
    loss_fn = nn.CrossEntropyLoss()
    probe.train()
    for step in range(60):
        logits = probe(x)
        loss = loss_fn(logits, y)
        # opt.zero_grad()        # <-- 🔧 BUG: this line is commented out. Restore it!
        loss.backward()
        opt.step()
        if step % 15 == 0:
            print(f"step {step:2d}: loss={loss.item():.4f}")  # should march toward 0
    print("If the loss did NOT fall, you found why: gradients accumulate without "
          "zero_grad().")

overfit_one_batch_demo()

## 6 · Evaluate, experiment & reflect

### Loss curves
Read the gap (L5): close + low = good; train falling while val rises = overfitting;
both high = underfitting. Use this to decide your *next* experiment.

In [ ]:
plot_curves(history)

### Confusion matrix on validation
A single accuracy number hides *which* classes get confused. Build the matrix on
**val** (not test) and read off the look-alikes — for CIFAR, expect cat↔dog and
automobile↔truck. (Text/tabular: same matrix; a language-model track would show
generated samples + perplexity instead.)

In [ ]:
@torch.no_grad()
def confusion_matrix(model, loader, n_classes):
    model.eval()
    cm = torch.zeros(n_classes, n_classes, dtype=torch.long)
    for x, y in loader:
        preds = model(x.to(device)).argmax(dim=-1).cpu()
        for t, p in zip(y, preds):
            cm[t, p] += 1
    return cm

cm = confusion_matrix(model, val_loader, len(classes))
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha="right")
ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion (val)")
for i in range(len(classes)):                    # annotate counts
    for j in range(len(classes)):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=7)
plt.tight_layout(); plt.show()

# Per-class recall (of the true class c, what fraction did we catch?)
recall = cm.diag() / cm.sum(dim=1).clamp(min=1)
for c, r in sorted(zip(classes, recall.tolist()), key=lambda t: t[1]):
    print(f"  recall  {c:<12} {r:.2%}")

### 🔧 Experiment — beat the baseline, then ablate

Run **at least two** modify-and-observe experiments, changing **one thing at a time**
and re-reading the curves. Fill in the table with **validation** numbers (test stays
sealed). Suggestions: toggle augmentation/dropout, change `lr` or `weight_decay`, add
a layer, widen the model.

| # | Change (vs. baseline) | Val accuracy | Train–val gap | Notes |
|---|---|---|---|---|
| 0 | baseline (starter model) | _fill in_ | _fill in_ | _the run above_ |
| 1 | 🔧 _e.g. +dropout 0.5_ | | | |
| 2 | 🔧 _e.g. lr 3e-3_ | | | |

**Target:** beat your track's stated baseline (A > 65%, B > 70%, C > 95%). State the
trivial baseline you compare against (majority-class accuracy) and your best **val**
number before you open the test set.

In [ ]:
# 🔧 Scratch cell for experiments. Re-instantiate model + re-run train() here,
# changing ONE knob at a time, and copy the val_acc into the ablation table above.
# Example:
#   model_v1 = CNN(n_classes=len(classes), p_drop=0.5)
#   hist_v1 = train(model_v1, train_loader, val_loader, epochs=8, lr=1e-3)
#   plot_curves(hist_v1)
pass

### ✅ Final test evaluation — run this **once**

This is the only time you touch the **test** set. Pick the model + hyperparameters you
settled on using validation, make sure that `model` is trained, and run this cell **a
single time**. The number it prints is the one you report. Re-running it after more
tuning is exactly the test-leakage the rubric penalises — so tune *first*, seal it
*last*.

In [ ]:
# ✅ ONE-SHOT held-out test evaluation. Do NOT tune after seeing this number.
test_loss, test_acc = evaluate(model, test_loader, nn.CrossEntropyLoss())
majority = max(
    collections.Counter(int(test_set[i][1]) for i in range(len(test_set))).values()
) / len(test_set)
print(f"majority-class baseline (test): {majority:.2%}")
print(f"FINAL TEST  loss={test_loss:.3f}  accuracy={test_acc:.2%}")
print("beat baseline:", "YES" if test_acc > majority else "no -- keep iterating")

## 7 · Rubric checklist

Before you submit, tick each box. These mirror the **Capstone rubric** — honest
methodology counts more than a high number.

- [ ] **Real dataset, real model.** A public dataset (cited) and a model *you* defined
      and trained — not a pre-trained model called as-is.
- [ ] **Honest train / val / test discipline.** A clean 3-way split; the test set was
      touched **exactly once**, at the end. No hyperparameter was chosen by looking at
      test.
- [ ] **A baseline, beaten.** You state a trivial baseline (majority-class /
      mean / linear) and your model beats it on the **held-out test** set.
- [ ] **At least one form of regularization** (dropout, weight decay, augmentation,
      early stopping) — and you can say *why*.
- [ ] **Loss curves + a confusion matrix / metric.** Train-vs-val curves you can read,
      plus a confusion matrix (or per-class metric / generated samples) — not just one
      accuracy number.
- [ ] **A fixed seed** so your numbers reproduce.
- [ ] **A short written report** (the template below), including one **honest failure
      case**.

## 8 · 🔧 Report template

Fill this in — it's the written deliverable. An honest, well-analysed result scores
higher than an inflated one.

**Dataset & task.** 🔧 _What dataset (source + size + your split sizes), and what are
you predicting?_

**Model & training choices.** 🔧 _Architecture (and why), loss, optimizer, learning
rate, regularization, epochs._

**Baseline.** 🔧 _The trivial baseline you compare against, and its score._

**Final test metric.** 🔧 _Your single held-out **test** number (e.g. accuracy / F1),
next to the baseline. Did you beat your target?_

**What I tried (ablation).** 🔧 _Paste your ablation table. Which change helped most,
and why? What did the train–val gap tell you?_

**What worked / what failed.** 🔧 _One honest failure case — a class the model confuses,
an example it gets wrong, a setting that made things worse._

**With more time.** 🔧 _One concrete thing you'd try next (more data, a deeper model,
a different architecture, better augmentation)._

---

That's the hand-off. Everything across the course — tensors, the training loop,
CNNs and RNNs/LSTMs, regularization, and reading loss curves — now lives in one place,
on a **real dataset you chose**, and is judged on a **held-out test set** you evaluate
exactly once. Make it small, make it finished, and tell the truth about the numbers.